In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 워크로드 사용량

<span id="usage"></span>
사용량은 워크로드를 위해 QPU가 잠겨 있는 시간을 측정한 값이며, 사용 중인 실행 모드에 따라 계산 방법이 다릅니다.

* 세션 사용량은 세션이 활성 상태인 동안의 실제 경과 시간입니다. 세션 상태 전환에 대한 자세한 내용은 [세션 길이](/guides/run-jobs-session#session-length)를 참조하세요.
* 배치 사용량은 배치 내 모든 작업의 양자 시간(QPU 복합체가 작업을 처리하는 데 소요된 시간)의 합계입니다.
* 단일 작업 사용량은 해당 작업이 처리 중에 사용하는 양자 시간입니다.

실패하거나 취소된 작업도 특정 상황에서는 사용량에 포함될 수 있으니 자세한 내용은 [실패 및 취소된 작업](#failed-job) 섹션을 참조하세요.

유료 플랜 사용자의 경우, 사용량에 따라 워크로드 비용이 결정됩니다. 자세한 내용은 [비용 관리](/guides/manage-cost)를 참조하세요.

<span id="failed-job"></span>
## 실패 및 취소된 작업의 사용량
작업이 실패하거나 취소된 경우, 보고되는 사용량은 다음과 같습니다:

* 작업 또는 배치 모드: 보고되는 사용량은 워크로드 실행을 위해 QPU가 잠긴 시점부터 실패하거나 취소된 시점까지의 시간입니다. 따라서 잠금 전에 실패하거나 취소된 경우 보고되는 사용량은 0입니다. 그렇지 않은 경우, 워크로드의 보고 사용량은 워크로드가 실패하거나 취소되기 전까지의 사용량입니다. 따라서 일부 실패한 작업은 보고된 사용량에 표시되지 않을 수 있고, 다른 작업은 표시될 수 있습니다.

* 세션 모드: 보고되는 사용량은 실패하거나 취소된 작업 수와 관계없이 세션이 활성 상태인 동안의 실제 경과 시간입니다.

<span id="view-usage"></span>
## 워크로드의 실제 사용량 조회
워크로드가 완료된 후, 실제 사용량을 확인하는 방법은 여러 가지가 있습니다:

- `qiskit-ibm-runtime` 0.30 이상에서 [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) 또는 [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage)를 실행합니다. 이전 버전의 `qiskit-ibm-runtime`(>= 0.23 및 < 0.30)을 사용하는 경우, `session.details()["usage_time"]` 및 `batch.details()["usage_time"]`에서 사용량을 확인할 수 있습니다.
- [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails)를 사용하여 특정 배치 또는 세션의 사용량을 확인합니다.
- [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById)를 사용하여 단일 작업의 사용량을 확인합니다.

<span id="instance-usage"></span>
## 인스턴스 사용량 보기
[인스턴스](https://quantum.cloud.ibm.com/instances) 페이지에서, 또는 적절한 권한이 있는 경우 [분석](https://quantum.cloud.ibm.com/analytics) 페이지에서 인스턴스의 사용량을 확인할 수 있습니다. 두 페이지는 사용량 계산 방식이 다르기 때문에 표시되는 사용량 수치가 다를 수 있습니다.

인스턴스 페이지는 현재 날짜의 현재 시간까지 최근 28일(롤링) 동안의 실시간 사용량을 표시합니다. 분석 페이지의 사용량은 매 시간 다시 계산되며 최근 28 전체 일을 포함합니다. 즉, 28일 전 00:00부터 오늘 정각까지의 사용량을 표시합니다.

## 작업 제출 전 사용량 추정
오류 억제 및 완화를 위한 추가 작업으로 인해 정확한 로컬 추정은 복잡하지만, 다음 기본 공식을 사용하여 예상 사용량의 근사치를 구할 수 있습니다:

`<per sub-job overhead> + (rep_delay + <circuit length>) * <num executions>`

- `<per sub-job overhead>`는 서브 작업당 약 2초의 오버헤드입니다. 여기에는 페이로드를 제어 전자장치에 로드하는 등의 작업이 포함됩니다. 기본 요소(primitive) 작업이 실행 엔진이 한 번에 처리하기에 너무 큰 경우, 여러 서브 작업으로 나뉠 수 있습니다.
- `rep_delay`는 [사용자가 커스터마이즈 가능한](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay) 옵션이며, 기본값은 `backend.default_rep_delay`로 주어지고, 대부분의 IBM Quantum Backend에서는 250마이크로초입니다. `rep_delay`를 낮추면 총 QPU 실행 시간이 단축되지만, 상태 준비 오류율이 증가한다는 점에 유의하세요. 자세한 내용은 [동적 반복률 실행](/guides/repetition-rate-execution) 가이드를 참조하세요.
- `<circuit length>`는 총 명령 길이입니다. 각 명령은 QPU에서 실행하는 데 걸리는 시간이 다르므로 총 길이는 Circuit마다 다릅니다. 예를 들어, 측정은 `x` Gate보다 56배 더 오래 걸릴 수 있습니다. `backend.target[<instruction>][<qubit>].duration`을 사용하여 각 명령의 정확한 지속 시간을 확인할 수 있습니다. 일반적인 Circuit 길이는 50-100마이크로초 사이일 가능성이 높습니다. 기본 요소(primitive)와 함께 오류 억제 또는 완화 기술을 사용하는 경우, Circuit에 추가 명령이 삽입되어 총 Circuit 길이가 늘어날 수 있습니다.
    > **Note:** [실험적 `scheduler_timing` 옵션](/guides/visualize-circuit-timing)은 총 Circuit 시간을 반환하지만, 이는 청구에 사용되는 시간이 아닙니다.
- `<num executions>`는 PUB 요소가 브로드캐스트된 후 생성된 Circuit을 기준으로, 총 Circuit 수에 샷 수를 곱한 값입니다.
    - 기본 요소(primitive)와 함께 오류 완화 기술을 사용하는 경우, 완화 프로세스의 일환으로 추가 Circuit이 실행될 수 있어 총 실행 횟수가 증가할 수 있습니다. 또한 PEA 및 PEC와 같은 고급 오류 완화 기술은 노이즈 학습을 위한 Circuit 실행이 필요하기 때문에 훨씬 높은 오버헤드가 발생합니다.
    - Estimator는 Qubit 단위로 교환 가능한 관측값을 그룹화하여 실행 횟수를 줄입니다.

고급 오류 완화 기술이나 커스텀 `rep_delay`를 사용하지 않는 경우, 빠른 공식으로 `2+0.00035*<num executions>`를 사용할 수 있습니다.

## 다음 단계
> **Tip:** - 다음 팁을 검토하세요: [작업 실행 시간 최소화](minimize-time).
>     - [최대 실행 시간](max-execution-time)을 설정하세요.
>     - [Transpile](./transpile/) 섹션에서 로컬 Transpiler 방법을 알아보세요.
>     - [Transpiler 설정 비교](/guides/circuit-transpilation-settings) 가이드를 시도해 보세요.